In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input, Multiply
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# --- PATHS (Reusing your existing structure) ---
BASE_DIR = r"C:\Users\avina\OneDrive\Desktop\DeepFake_Sentinel\dataset"
PROCESSED_DIR = os.path.join(BASE_DIR, "processed_images")
MODELS_DIR = r"C:\Users\avina\OneDrive\Desktop\DeepFake_Sentinel\models"

# Configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_INITIAL = 10
EPOCHS_FINETUNE = 5

print(f"✅ Setup complete. Data path: {PROCESSED_DIR}")

✅ Setup complete. Data path: C:\Users\avina\OneDrive\Desktop\DeepFake_Sentinel\dataset\processed_images


In [2]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2 # Keeping your 80/20 split
)

train_generator = train_datagen.flow_from_directory(
    PROCESSED_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    PROCESSED_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Found 8000 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.


In [ ]:
def build_attention_model():
    # Base MobileNetV2
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False 
    
    inputs = Input(shape=(224, 224, 3))
    x = base_model(inputs)
    
    # --- HYBRID ATTENTION LAYER ---
    gap = GlobalAveragePooling2D()(x)
    
    # Learning 'where' to look for deepfake artifacts
    sq = Dense(1280 // 16, activation='relu')(gap)
    ex = Dense(1280, activation='sigmoid')(sq)
    attended = Multiply()([gap, ex]) 
    
    # Dense Classifier Head
    x = Dense(512, activation='relu')(attended)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    
    return Model(inputs, outputs)

model = build_attention_model()
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 7,      │  2,257,984 │ input_layer_1[0]… │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 80)        │    102,480 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1280)      │    103,680 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 1280)      │          0 │ global_average_p… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 512)       │    655,872 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │        513 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,120,529 (11.90 MB)

 Trainable params: 862,545 (3.29 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [4]:
print("🚀 Phase 1: Training Attention Head...")
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS_INITIAL
)

🚀 Phase 1: Training Attention Head...
Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 201s 790ms/step - accuracy: 0.6492 - loss: 0.6232 - val_accuracy: 0.7095 - val_loss: 0.5689
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 200s 797ms/step - accuracy: 0.6969 - loss: 0.5839 - val_accuracy: 0.7355 - val_loss: 0.5385
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 168s 673ms/step - accuracy: 0.7089 - loss: 0.5642 - val_accuracy: 0.7230 - val_loss: 0.5413
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 166s 664ms/step - accuracy: 0.7182 - loss: 0.5519 - val_accuracy: 0.7095 - val_loss: 0.5486
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 164s 657ms/step - accuracy: 0.7343 - loss: 0.5399 - val_accuracy: 0.7415 - val_loss: 0.5220
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 167s 667ms/step - accuracy: 0.7345 - loss: 0.5319 - val_accuracy: 0.7425 - val_loss: 0.5216
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 175s 701ms/step - accuracy: 0.7399 - loss: 0.5246 - val_accuracy: 0.7510 - val_loss: 0.5059
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━

In [5]:
print("🔓 Unfreezing Backbone for Full Deep Learning Fine-tuning...")
model.layers[1].trainable = True # Unfreeze MobileNetV2

model.compile(
    optimizer=Adam(learning_rate=1e-5), # Lower rate to preserve weights
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS_FINETUNE
)

# Save the 100% DL Model
os.makedirs(MODELS_DIR, exist_ok=True)
final_path = os.path.join(MODELS_DIR, "deepfake_sentinel_v2_attention.h5")
model.save(final_path)
print(f"🎉 SUCCESS! 100% Deep Learning Model saved at: {final_path}")

🔓 Unfreezing Backbone for Full Deep Learning Fine-tuning...
Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 635s 2s/step - accuracy: 0.6747 - loss: 0.6210 - val_accuracy: 0.5100 - val_loss: 1.6932
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 605s 2s/step - accuracy: 0.7554 - loss: 0.5077 - val_accuracy: 0.5145 - val_loss: 1.7707
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 592s 2s/step - accuracy: 0.7884 - loss: 0.4557 - val_accuracy: 0.5560 - val_loss: 1.3763
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 582s 2s/step - accuracy: 0.8149 - loss: 0.4040 - val_accuracy: 0.6165 - val_loss: 1.0385
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 580s 2s/step - accuracy: 0.8379 - loss: 0.3694 - val_accuracy: 0.6880 - val_loss: 0.8106


🎉 SUCCESS! 100% Deep Learning Model saved at: C:\Users\avina\OneDrive\Desktop\DeepFake_Sentinel\models\deepfake_sentinel_v2_attention.h5
